# Mini Project 5 — IT Helpdesk LLM System
Pipeline: `ask_query` → `classify_query` → `response_generate` → save to JSON

In [ ]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

AZURE_OPENAI_API_KEY     = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT    = os.getenv("AZURE_OPENAI_ENDPOINT")   # ← bug fix: was reading API_KEY
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

AZURE_CHAT_SEARCH      = os.getenv("AZURE_CHAT_SEARCH")
AZURE_CHAT_CLASSIFIER  = os.getenv("AZURE_CHAT_CLASSIFIER")

for name, value in {
    "AZURE_OPENAI_API_KEY":     AZURE_OPENAI_API_KEY,
    "AZURE_OPENAI_ENDPOINT":    AZURE_OPENAI_ENDPOINT,
    "AZURE_OPENAI_API_VERSION": AZURE_OPENAI_API_VERSION,
    "AZURE_CHAT_SEARCH":        AZURE_CHAT_SEARCH,
    "AZURE_CHAT_CLASSIFIER":    AZURE_CHAT_CLASSIFIER,
}.items():
    print(f"{name:30s} -> {'OK' if value else 'MISSING'}")

In [ ]:
import pandas as pd

df = pd.read_csv("tickets_IT_helpdesk.csv")

df.head()

## Step 1 — Initialise LLM clients
Two separate Azure deployments:
- **classifier** (`AZURE_CHAT_CLASSIFIER`) — lightweight model used only for classification
- **search / responder** (`AZURE_CHAT_SEARCH`) — full model used for answer generation

In [ ]:
from openai import AzureOpenAI

# Shared client — deployment name is passed per-call
client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
)

print("AzureOpenAI client ready.")

## Step 2 — `ask_query`
Simple helper that prompts the user for their IT issue.

In [ ]:
def ask_query() -> str:
    """
    Prompt the user to describe their IT problem.
    Returns the raw query string.
    """
    query = input("Describe your IT issue: ").strip()
    if not query:
        raise ValueError("Query cannot be empty.")
    return query

# Quick smoke-test (comment out when running the full pipeline)
# query = ask_query()
# print("User query:", query)

## Step 3 — `classify_query`
Calls the classifier deployment and returns one of: **Access | Network | Hardware | Software**.

In [ ]:
VALID_CATEGORIES = {"Access", "Network", "Hardware", "Software"}

def classify_query(query: str) -> str:
    """
    Classify the user query into one of the four IT categories.
    Returns a string: 'Access' | 'Network' | 'Hardware' | 'Software'.
    """
    system_prompt = (
        "You are an IT helpdesk ticket classifier. "
        "Given a user's IT issue, respond with exactly ONE word from this list: "
        "Access, Network, Hardware, Software. "
        "Do NOT include any other text, punctuation, or explanation."
    )

    response = client.chat.completions.create(
        model=AZURE_CHAT_CLASSIFIER,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": query},
        ],
        temperature=0,       # deterministic for classification
        max_tokens=10,
    )

    raw = response.choices[0].message.content.strip()

    # Normalise capitalisation and validate
    category = raw.capitalize()
    if category not in VALID_CATEGORIES:
        print(f"[WARN] Unexpected category '{raw}', defaulting to 'Software'")
        category = "Software"

    return category

# Quick smoke-test
# print(classify_query("Laptop saya tidak bisa menyala sama sekali"))

## Step 4 — `response_generate`
Retrieves relevant tickets from the CSV (same category) and uses them as context for the answer.

In [ ]:
def retrieve_context(category: str, top_k: int = 3) -> list[dict]:
    """
    Filter the ticket dataframe by category and return top_k tickets
    as a list of dicts with keys: ticket_id, issue, expected_resolution.
    """
    filtered = df[df["category"] == category].head(top_k)
    return filtered[["ticket-id", "issue", "expected_resolution"]].rename(
        columns={"ticket-id": "ticket_id"}
    ).to_dict(orient="records")


def build_context_text(tickets: list[dict]) -> str:
    """Format retrieved tickets into a readable context block."""
    lines = []
    for t in tickets:
        lines.append(
            f"[{t['ticket_id']}]\n"
            f"Issue    : {t['issue']}\n"
            f"Resolution: {t['expected_resolution']}"
        )
    return "\n\n".join(lines)


def response_generate(query: str, category: str) -> tuple[str, list[dict]]:
    """
    Retrieve context tickets, then generate an answer grounded in those tickets.
    Returns (response_text, context_tickets).
    """
    context_tickets = retrieve_context(category)
    context_text    = build_context_text(context_tickets)

    system_prompt = (
        "You are a helpful IT helpdesk agent. "
        "Use the reference tickets below to answer the user's question. "
        "Be concise and actionable. If the exact problem is not in the tickets, "
        "give the best advice based on similar issues.\n\n"
        f"=== Reference Tickets (category: {category}) ===\n"
        f"{context_text}\n"
        "==="
    )

    response = client.chat.completions.create(
        model=AZURE_CHAT_SEARCH,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": query},
        ],
        temperature=0.3,
        max_tokens=512,
    )

    answer = response.choices[0].message.content.strip()
    return answer, context_tickets

# Quick smoke-test
# ans, ctx = response_generate("Saya tidak bisa login VPN", "Network")
# print(ans)

## Step 5 — Full pipeline + save result to JSON

In [ ]:
import datetime
import pathlib

OUTPUT_FILE = pathlib.Path("helpdesk_results.json")

def run_pipeline(save: bool = True) -> dict:
    """
    End-to-end pipeline:
      ask_query → classify_query → response_generate → save to JSON
    Returns the result dict.
    """
    # 1. Collect query
    query = ask_query()
    print(f"\n[Query]    {query}")

    # 2. Classify
    category = classify_query(query)
    print(f"[Category] {category}")

    # 3. Generate response
    response, context = response_generate(query, category)
    print(f"\n[Response]\n{response}")

    # 4. Build result record
    result = {
        "timestamp": datetime.datetime.now().isoformat(),
        "query":     query,
        "category":  category,
        "response":  response,
        "context":   context,
    }

    # 5. Append to JSON file
    if save:
        existing = []
        if OUTPUT_FILE.exists():
            with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
                existing = json.load(f)

        existing.append(result)

        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            json.dump(existing, f, ensure_ascii=False, indent=2)

        print(f"\n[Saved]    Result appended to {OUTPUT_FILE}")

    return result


# ── Run ──────────────────────────────────────────────────────────────────────
result = run_pipeline(save=True)